# 03 Reproducing the Taillade et al. (2023) model values

Before reading anything into real Biomass data I wanted to know the forward model behaves as published (report Section 4.2). This notebook checks, in order:

1. the Klein and Swift seawater permittivity, against textbook values at L-band and the El Niño value quoted in the report
2. the salinity sensitivities in Table I of Taillade et al. (2023), with the pure and tilted Bragg models
3. how big the salinity signal really is across a river plume, and how many pixels it would take to see it
4. the Elfouhaily wave spectrum, and what the waves Biomass sees look like next to the wind sea
5. how wind leaks back into the HH/VV ratio and CPD once the Bragg waves are tilted

No data needed, it's all physics, and the functions live in `bm_tools.py` so the other notebooks use exactly the same code.

In [ ]:
# ---- settings ----
from pathlib import Path
OUT = Path(r"C:\Users\Orlaith.Doyle\biomass_processed\model_checks")

F_P, F_L, F_C = 435e6, 1.26e9, 5.405e9       # Biomass, ROSE-L, Sentinel-1
T_TABLE = 25.0                                # Table I matches at 25 C ...
S_LO, S_HI = 20.0, 39.0                       # ... and only over 20 to 39 psu, not the 20 to 30 in the caption
ANGLES = [30, 35, 40, 45, 50, 55, 60]
WIND = 7.0                                    # m/s for the tilted model

# fill these in from Table I if you want the % difference printed, {(band, angle): dB per psu}
PUBLISHED_PURE = {}
PUBLISHED_TILTED = {}

In [ ]:
import sys
sys.path.append(str(Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bm_tools as bm

OUT.mkdir(parents=True, exist_ok=True)

def save(fig, what):
    fig.savefig(OUT / f"{what}.png", dpi=200, bbox_inches="tight")
    print("saved", what + ".png")

## 1. Permittivity

Klein and Swift (1977) model seawater as one Debye relaxation plus an ionic conductivity term, eps = eps' - i eps''. The conductivity term scales as 1/omega, so at 435 MHz it's about three times bigger than the real part, and its salinity dependence is roughly three times bigger than at 1.4 GHz. That was the original reason for trying P-band for salinity.

Expected: about 72 - 66i at L-band for open ocean (35 psu, 20 C), and about 69.6 - 240.8i for the El Niño scene (34.5 psu, 30.4 C) at P-band.

In [ ]:
for lab, S, T, f in [("L-band, 35 psu 20 C", 35, 20, 1.413e9),
                     ("P-band, El Nino 34.5 psu 30.4 C", 34.5, 30.4, F_P),
                     ("P-band, plume 15 psu 28 C", 15, 28, F_P)]:
    e = bm.eps_klein_swift(S, T, f)
    print(f"{lab:35s} eps = {e.real:6.1f} {e.imag:+7.1f}i")

# how much more salinity moves eps'' at P than at L
dP = abs(bm.eps_klein_swift(36, 20, F_P).imag - bm.eps_klein_swift(34, 20, F_P).imag)
dL = abs(bm.eps_klein_swift(36, 20, 1.413e9).imag - bm.eps_klein_swift(34, 20, 1.413e9).imag)
print("d eps'' / d S at P-band is %.1f x L-band" % (dP / dL))

## 2. Table I sensitivities

Taillade et al. give the change in VV backscatter per psu at P, L and C band over 30 to 60 deg. The pure Bragg values come straight from Eq. 14 (the wave spectrum cancels in the dB difference so wind doesn't matter). The tilted values use the two scale model (Eq. 19) with long wave slopes from the Elfouhaily spectrum.

What I found: the pure Bragg numbers match exactly at 25 C, but only if the sensitivity is taken over 20 to 39 psu rather than the 20 to 30 psu in the table caption. The tilted ones agree within 1.4 %. My tilted model splits the slope variance evenly between the along-look and cross-look directions, which is a simplification, so small differences there are expected.

In [ ]:
def sens(model, f, th):
    lo = model(bm.eps_klein_swift(S_LO, T_TABLE, f), th, f, WIND)[1]
    hi = model(bm.eps_klein_swift(S_HI, T_TABLE, f), th, f, WIND)[1]
    return (bm.db(hi) - bm.db(lo)) / (S_HI - S_LO)

rows = []
for band, f in [("P", F_P), ("L", F_L), ("C", F_C)]:
    for th in ANGLES:
        rows.append({"band": band, "angle": th,
                     "pure dB/psu": sens(bm.pure_bragg, f, th),
                     "tilted dB/psu": sens(bm.tilted_bragg, f, th)})
tab = pd.DataFrame(rows)

def pct(row, pub, col):
    v = pub.get((row["band"], row["angle"]))
    return np.nan if v is None else 100 * (row[col] - v) / v

tab["pure vs paper %"] = tab.apply(lambda r: pct(r, PUBLISHED_PURE, "pure dB/psu"), axis=1)
tab["tilted vs paper %"] = tab.apply(lambda r: pct(r, PUBLISHED_TILTED, "tilted dB/psu"), axis=1)
tab.to_csv(OUT / "table1_reproduction.csv", index=False)
print(tab.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
for band, c in zip("PLC", ["tab:green", "tab:blue", "tab:red"]):
    t = tab[tab.band == band]
    ax.plot(t.angle, t["pure dB/psu"], "o-", color=c, label=f"{band} pure")
    ax.plot(t.angle, t["tilted dB/psu"], "s--", color=c, label=f"{band} tilted")
ax.set_xlabel("incidence (deg)"); ax.set_ylabel("VV sensitivity (dB per psu)")
ax.set_title(f"T = {T_TABLE} C, {S_LO:.0f} to {S_HI:.0f} psu, U10 = {WIND} m/s")
ax.legend(ncol=3, fontsize=8)
save(fig, "table1_sensitivity")
plt.show()

## 3. How big is the salinity signal in a plume?

A big change in eps doesn't mean a big change in what the radar sees. Seawater is already so reflective at P-band that g_HH and g_VV barely move. This takes a drop from 30 to 15 psu at 28 C, roughly across the Amazon plume, over the Biomass incidence range.

Expected from the report: HH/VV 0.06 to 0.13 dB, CPD 0.02 to 0.04 deg, VV about 0.4 dB.

Then Eq. 6 says how many independent pixels have to be averaged to measure a CPD change that small. At coherence 0.95 and 0.03 deg it's about 2 x 10^5, an area several km across, and over that kind of area the wind and swell change too.

A note on the CPD sign. With eps written as eps' - i eps'' (the report's convention) the Bragg CPD over the ocean comes out at about +1 deg. Writing eps with +i eps'' instead flips it to about -1.3 deg, which is the number I quoted for El Niño earlier on. Same physics, only the time convention changes, so check which one a paper uses before comparing signs.

In [ ]:
f = F_P
for th in [23, 28.5, 34]:
    a = bm.pure_bragg(bm.eps_klein_swift(30, 28, f), th, f, WIND)
    b = bm.pure_bragg(bm.eps_klein_swift(15, 28, f), th, f, WIND)
    print(f"theta {th:4.1f}: HH/VV {bm.db(b[0] / b[1]) - bm.db(a[0] / a[1]):+.3f} dB, "
          f"CPD {b[2] - a[2]:+.4f} deg, VV {bm.db(b[1]) - bm.db(a[1]):+.2f} dB")

print("\nCPD at El Nino (34.5 psu, 30.4 C, 29.3 deg): %+.2f deg"
      % bm.pure_bragg(bm.eps_klein_swift(34.5, 30.4, f), 29.28, f, 3.7)[2])

def n_needed(sig_deg, coh):
    s = np.radians(sig_deg)
    return (1 - coh**2) / (2 * coh**2 * s**2)

print("\npixels needed for a 0.03 deg CPD precision:")
for coh in (0.90, 0.95, 0.98):
    print(f"  coherence {coh}: {n_needed(0.03, coh):.1e}")
print("check eq 6: 25 looks at 0.95 -> %.1f deg" % bm.cpd_std_deg(25, 0.95))

fig, ax = plt.subplots(figsize=(6, 4))
N = np.logspace(0, 6, 200)
for coh in (0.9, 0.95, 0.98):
    ax.loglog(N, bm.cpd_std_deg(N, coh), label=f"|gamma| = {coh}")
ax.axhline(0.03, color="k", ls=":", label="plume CPD signal")
ax.set_xlabel("number of looks N"); ax.set_ylabel("CPD std (deg)")
ax.legend(); ax.grid(alpha=0.3, which="both")
save(fig, "cpd_precision")
plt.show()

## 4. The Elfouhaily spectrum and the waves Biomass sees

From the wind speed alone the Elfouhaily model gives the whole wind sea spectrum, from the longest waves down to ripples. The peak is at k_p = g Omega_c^2 / U10^2, so for a fully developed sea (Omega_c = 0.84) the peak wavelength is about 25 m at 5 m/s and about 90 m at 10 m/s.

Biomass's Bragg waves (shaded) sit at 0.6 to 0.9 m, much longer than the few cm C-band sees. That's why its backscatter should depend less on the wind.

Plotted as the curvature spectrum B(k) = k^3 S(k), the usual way, since S(k) itself drops over many decades.

In [ ]:
k = np.logspace(-3, 3, 3000)
fig, ax = plt.subplots(figsize=(7, 4.5))
for U in [3, 5, 7, 10, 15, 20]:
    S = bm.elfouhaily(k, U)
    ax.loglog(k, k**3 * S, label=f"{U} m/s")
    print(f"U10 = {U:2d} m/s: peak wavelength {2 * np.pi / k[np.argmax(S)]:6.1f} m")
kb = 2 * (2 * np.pi * F_P / bm.C) * np.sin(np.radians([23, 34]))
ax.axvspan(kb.min(), kb.max(), color="tab:green", alpha=0.25, label="Biomass Bragg (0.6 to 0.9 m)")
kc = 2 * (2 * np.pi * F_C / bm.C) * np.sin(np.radians([30, 45]))
ax.axvspan(kc.min(), kc.max(), color="tab:red", alpha=0.2, label="C-band Bragg")
ax.set_ylim(1e-5, 1e-1)
ax.set_xlabel("wavenumber k (rad/m)"); ax.set_ylabel("B(k) = k$^3$ S(k)")
ax.legend(fontsize=8, ncol=2)
ax.set_title("Elfouhaily omnidirectional spectrum, fully developed sea")
save(fig, "elfouhaily")
plt.show()

## 5. Wind comes back in once the waves are tilted

In the pure Bragg model wind cancels exactly in HH/VV and the CPD. Once the Bragg patches ride on long waves it doesn't any more: tilting along the look direction changes the local incidence angle, and tilting across it rotates the patch, which mixes some VV into HH and the other way round. The steeper the long waves (more wind), the more alike the channels get.

This plots how much HH/VV, CPD and VV change with wind next to the size of the plume salinity signal from section 3. The wind effect is bigger, which is the whole story of notebook 04 in one plot.

In [ ]:
th = 30
winds = np.linspace(2, 20, 19)
e35 = bm.eps_klein_swift(35, 28, F_P)
res = np.array([bm.tilted_bragg(e35, th, F_P, U) for U in winds])
pure = bm.pure_bragg(e35, th, F_P, 7)

fig, axs = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
axs[0].plot(winds, bm.db(res[:, 1]), "o-"); axs[0].set_ylabel("VV sigma0 (dB)")
axs[1].plot(winds, bm.db(res[:, 0] / res[:, 1]), "o-")
axs[1].axhline(bm.db(pure[0] / pure[1]), color="k", ls="--", label="pure Bragg")
axs[1].set_ylabel("HH/VV (dB)")
axs[2].plot(winds, res[:, 2], "o-")
axs[2].axhline(pure[2], color="k", ls="--", label="pure Bragg")
axs[2].set_ylabel("CPD (deg)")
for a in axs:
    a.set_xlabel("U10 (m/s)"); a.grid(alpha=0.3)
axs[1].legend(); axs[2].legend()
fig.suptitle(f"tilted Bragg at {th} deg, 35 psu, 28 C")
save(fig, "tilted_vs_wind")
plt.show()

print("HH/VV change 5 -> 10 m/s: %.2f dB  (plume salinity: ~0.1 dB)"
      % (bm.db(res[8, 0] / res[8, 1]) - bm.db(res[3, 0] / res[3, 1])))
print("CPD change   5 -> 10 m/s: %.2f deg (plume salinity: ~0.03 deg)" % (res[8, 2] - res[3, 2]))